# Fairness, Stability & Stress Testing

Evaluates risk, robustness, constraints. Not optimizing fairness—it's a constraint.

Principles: fairness is a constraint (not a target), stability beats marginal lift, decisions matter more than metrics.

Goal: find where the policy breaks, where fairness risks appear, what constraints production needs.

Models: Baseline Logistic Regression (Step 7), LightGBM (Step 8).

Policy: fixed threshold PD < 0.04 (from Step 6).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

# Apply temporal splits
train_ids = df[df['ID'] <= 20000]['ID'].values
val_ids = df[(df['ID'] > 20000) & (df['ID'] <= 25000)]['ID'].values
test_ids = df[df['ID'] > 25000]['ID'].values

train_df = df[df['ID'].isin(train_ids)].copy()
val_df = df[df['ID'].isin(val_ids)].copy()
test_df = df[df['ID'].isin(test_ids)].copy()

target_col = 'default payment next month'

# Use feature set from Step 7 (without PAY_0)
approved_features = [
    'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'LIMIT_BAL',
    'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

# Policy threshold from Step 6
LGD_BASELINE = 0.50
OPPORTUNITY_COST_RATIO = 0.02
pd_threshold = OPPORTUNITY_COST_RATIO / LGD_BASELINE

print(f"Test set size: {len(test_df):,}")
print(f"Policy threshold: PD < {pd_threshold:.4f}")
print(f"Features: {len(approved_features)} (PAY_0 excluded from Step 7)")

## Step 1: Group Definitions

Identify subgroups from available attributes. Important: these are proxies with limitations—sex is binary (dataset limitation), age buckets are arbitrary, education levels may not capture real differences.

In [ ]:
# Define groups for fairness analysis
print("=== GROUP DEFINITIONS ===\n")

# Group 1: Sex (binary proxy - note: this is a limitation)
test_df['group_sex'] = test_df['SEX'].map({1: 'Male', 2: 'Female'})
print("Group: Sex")
print(test_df['group_sex'].value_counts().sort_index())
print()

# Group 2: Age buckets
test_df['group_age'] = pd.cut(
    test_df['AGE'], 
    bins=[0, 30, 40, 50, 100], 
    labels=['<30', '30-40', '40-50', '50+']
)
print("Group: Age buckets")
print(test_df['group_age'].value_counts().sort_index())
print()

# Group 3: Education level
test_df['group_education'] = test_df['EDUCATION'].map({
    1: 'Graduate School',
    2: 'University',
    3: 'High School',
    4: 'Others',
    5: 'Unknown',
    6: 'Unknown',
    0: 'Unknown'
})
print("Group: Education")
print(test_df['group_education'].value_counts().sort_index())
print()

# Group 4: Credit limit buckets (socioeconomic proxy)
test_df['group_limit'] = pd.qcut(
    test_df['LIMIT_BAL'], 
    q=3, 
    labels=['Low', 'Medium', 'High']
)
print("Group: Credit Limit (tertiles)")
print(test_df['group_limit'].value_counts().sort_index())
print()

# Limitations of proxy-based analysis
print("=== LIMITATIONS ===\n")
print("1. **Sex is a binary proxy**: Does not capture gender identity or intersectionality")
print("2. **Age buckets are arbitrary**: May mask non-linear relationships")
print("3. **Education is self-reported**: May not reflect actual socioeconomic status")
print("4. **Credit limit is a proxy**: Correlated with income but not equivalent")
print("5. **No intersectional analysis**: Groups are analyzed separately, not in combination")
print("6. **Dataset limitations**: UCI dataset may not represent current population")
print("\n**Note**: This analysis identifies potential disparities but does not establish causality.")

## Step 2: Fairness Diagnostics

For each subgroup, compute:
- Approval rate
- Expected loss
- Default rate among approved loans

Compare both models (baseline logistic regression and LightGBM).

In [ ]:
# Recreate baseline model predictions (from Step 7)
print("=== LOADING MODEL PREDICTIONS ===\n")

# Prepare data (using refined labels from Step 7)
borderline_mask = (
    ((train_df['PAY_0'] > 0) & (train_df[target_col] == 0)) |
    ((train_df['PAY_0'] <= 0) & (train_df[target_col] == 1))
)
train_df_refined = train_df[~borderline_mask].copy()

X_train = train_df_refined[approved_features].copy()
y_train = train_df_refined[target_col].copy()
X_test = test_df[approved_features].copy()
y_test = test_df[target_col].copy()

# Baseline logistic regression
scaler_base = StandardScaler()
X_train_base_scaled = scaler_base.fit_transform(X_train)
X_test_base_scaled = scaler_base.transform(X_test)
X_train_base_scaled_df = pd.DataFrame(X_train_base_scaled, columns=approved_features, index=X_train.index)
X_test_base_scaled_df = pd.DataFrame(X_test_base_scaled, columns=approved_features, index=X_test.index)

default_rate_train = y_train.mean()
class_weight_ratio = (1 - default_rate_train) / default_rate_train
class_weights_base = {0: 1.0, 1: class_weight_ratio}

model_base = LogisticRegression(
    class_weight=class_weights_base,
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)
model_base.fit(X_train_base_scaled_df, y_train)
y_test_base_proba = model_base.predict_proba(X_test_base_scaled_df)[:, 1]

# LightGBM model (from Step 8)
monotonic_constraints = {
    'PAY_2': 1, 'PAY_3': 1, 'PAY_4': 1, 'PAY_5': 1, 'PAY_6': 1,
    'LIMIT_BAL': -1, 'AGE': -1,
    'PAY_AMT1': -1, 'PAY_AMT2': -1, 'PAY_AMT3': -1, 'PAY_AMT4': -1, 'PAY_AMT5': -1, 'PAY_AMT6': -1,
    'BILL_AMT1': 1, 'BILL_AMT2': 1, 'BILL_AMT3': 1, 'BILL_AMT4': 1, 'BILL_AMT5': 1, 'BILL_AMT6': 1,
}
monotonic_list = [monotonic_constraints.get(feat, 0) for feat in approved_features]

train_data = lgb.Dataset(
    X_train, 
    label=y_train,
    weight=np.where(y_train == 1, class_weight_ratio, 1.0)
)
val_data = lgb.Dataset(val_df[approved_features], label=val_df[target_col], reference=train_data)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_data_in_leaf': 20,
    'max_depth': 5,
    'monotone_constraints': monotonic_list,
    'verbosity': -1,
    'seed': 42
}

model_lgb = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=200,
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=0)]
)

y_test_lgb_proba = model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration)

print("✓ Models loaded and predictions generated")
print(f"  Baseline PD range: [{y_test_base_proba.min():.4f}, {y_test_base_proba.max():.4f}]")
print(f"  LightGBM PD range: [{y_test_lgb_proba.min():.4f}, {y_test_lgb_proba.max():.4f}]")

In [ ]:
# Function to compute fairness metrics by group
def compute_fairness_metrics(df_group, pd_values, model_name):
    """Compute approval rate, expected loss, and default rate by group."""
    df_group = df_group.copy()
    df_group['PD'] = pd_values
    df_group['EAD'] = df_group['LIMIT_BAL']
    df_group['approved'] = df_group['PD'] < pd_threshold
    df_group['expected_loss'] = df_group['PD'] * df_group['EAD'] * LGD_BASELINE
    
    approved = df_group[df_group['approved']]
    
    metrics = {
        'n_total': len(df_group),
        'n_approved': len(approved),
        'approval_rate': df_group['approved'].mean(),
        'total_expected_loss': approved['expected_loss'].sum() if len(approved) > 0 else 0,
        'mean_expected_loss_per_approved': approved['expected_loss'].mean() if len(approved) > 0 else 0,
        'default_rate_approved': approved[target_col].mean() if len(approved) > 0 else 0,
        'mean_pd': df_group['PD'].mean(),
        'mean_pd_approved': approved['PD'].mean() if len(approved) > 0 else 0
    }
    
    return metrics

# Compute metrics by group for both models
def analyze_group_fairness(group_col, group_name):
    """Analyze fairness metrics for a given group column."""
    print(f"\n{'='*60}")
    print(f"FAIRNESS ANALYSIS: {group_name}")
    print(f"{'='*60}\n")
    
    results_base = {}
    results_lgb = {}
    
    for group_val in test_df[group_col].dropna().unique():
        group_df = test_df[test_df[group_col] == group_val]
        
        # Get predictions for this group
        group_indices = group_df.index
        base_pd = y_test_base_proba[test_df.index.get_indexer(group_indices)]
        lgb_pd = y_test_lgb_proba[test_df.index.get_indexer(group_indices)]
        
        results_base[group_val] = compute_fairness_metrics(group_df, base_pd, 'Baseline')
        results_lgb[group_val] = compute_fairness_metrics(group_df, lgb_pd, 'LightGBM')
    
    # Create comparison table
    comparison_data = []
    for group_val in sorted(results_base.keys()):
        base = results_base[group_val]
        lgb = results_lgb[group_val]
        
        comparison_data.append({
            'Group': group_val,
            'N': base['n_total'],
            'Approval Rate (Base)': f"{base['approval_rate']:.1%}",
            'Approval Rate (LGB)': f"{lgb['approval_rate']:.1%}",
            'EL Total (Base)': f"${base['total_expected_loss']:,.0f}",
            'EL Total (LGB)': f"${lgb['total_expected_loss']:,.0f}",
            'Default Rate Approved (Base)': f"{base['default_rate_approved']:.1%}",
            'Default Rate Approved (LGB)': f"{lgb['default_rate_approved']:.1%}",
            'Mean PD (Base)': f"{base['mean_pd']:.4f}",
            'Mean PD (LGB)': f"{lgb['mean_pd']:.4f}"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))
    
    return results_base, results_lgb

# Analyze each group
results_sex_base, results_sex_lgb = analyze_group_fairness('group_sex', 'Sex')
results_age_base, results_age_lgb = analyze_group_fairness('group_age', 'Age Buckets')
results_edu_base, results_edu_lgb = analyze_group_fairness('group_education', 'Education')
results_limit_base, results_limit_lgb = analyze_group_fairness('group_limit', 'Credit Limit')

## Step 3: Disparity Assessment

Quantify disparities (absolute and relative), identify policy-driven vs model-driven disparities, and discuss tradeoffs.

In [ ]:
# Quantify disparities
print("=== DISPARITY ASSESSMENT ===\n")

def compute_disparities(results_dict, metric_name, model_name):
    """Compute absolute and relative disparities for a given metric."""
    values = {k: v[metric_name] for k, v in results_dict.items()}
    
    if len(values) == 0:
        return None
    
    max_val = max(values.values())
    min_val = min(values.values())
    mean_val = np.mean(list(values.values()))
    
    absolute_disparity = max_val - min_val
    relative_disparity = (max_val - min_val) / mean_val if mean_val > 0 else 0
    
    print(f"\n{model_name} - {metric_name}:")
    print(f"  Range: [{min_val:.4f}, {max_val:.4f}]")
    print(f"  Absolute disparity: {absolute_disparity:.4f}")
    print(f"  Relative disparity: {relative_disparity:.1%}")
    
    return {
        'min': min_val,
        'max': max_val,
        'mean': mean_val,
        'absolute_disparity': absolute_disparity,
        'relative_disparity': relative_disparity
    }

# Analyze disparities for approval rate (key fairness metric)
print("1. APPROVAL RATE DISPARITIES\n")
print("-" * 60)

disparities_sex_base = compute_disparities(results_sex_base, 'approval_rate', 'Baseline (Sex)')
disparities_sex_lgb = compute_disparities(results_sex_lgb, 'approval_rate', 'LightGBM (Sex)')

disparities_age_base = compute_disparities(results_age_base, 'approval_rate', 'Baseline (Age)')
disparities_age_lgb = compute_disparities(results_age_lgb, 'approval_rate', 'LightGBM (Age)')

disparities_limit_base = compute_disparities(results_limit_base, 'approval_rate', 'Baseline (Credit Limit)')
disparities_limit_lgb = compute_disparities(results_limit_lgb, 'approval_rate', 'LightGBM (Credit Limit)')

# Analyze disparities for expected loss
print("\n\n2. EXPECTED LOSS DISPARITIES (per approved loan)\n")
print("-" * 60)

disparities_el_sex_base = compute_disparities(results_sex_base, 'mean_expected_loss_per_approved', 'Baseline (Sex)')
disparities_el_sex_lgb = compute_disparities(results_sex_lgb, 'mean_expected_loss_per_approved', 'LightGBM (Sex)')

disparities_el_age_base = compute_disparities(results_age_base, 'mean_expected_loss_per_approved', 'Baseline (Age)')
disparities_el_age_lgb = compute_disparities(results_age_lgb, 'mean_expected_loss_per_approved', 'LightGBM (Age)')

# Policy-driven vs Model-driven disparities
print("\n\n3. POLICY-DRIVEN VS MODEL-DRIVEN DISPARITIES\n")
print("-" * 60)
print("""
**Policy-driven disparities**:
- Arise from the fixed threshold policy (PD < 0.04)
- Groups with higher mean PD will have lower approval rates
- This is expected if risk differences are real

**Model-driven disparities**:
- Arise from how models estimate PD differently across groups
- If models are well-calibrated, PD differences should reflect true risk
- If models are miscalibrated by group, this creates unfairness

**Key question**: Are approval rate differences justified by risk differences?
""")

# Compare mean PD vs approval rate
print("\n4. RISK-BASED JUSTIFICATION CHECK\n")
print("-" * 60)

for group_name, results_base, results_lgb in [
    ('Sex', results_sex_base, results_sex_lgb),
    ('Age', results_age_base, results_age_lgb),
    ('Credit Limit', results_limit_base, results_limit_lgb)
]:
    print(f"\n{group_name}:")
    print("  Group | Mean PD (Base) | Approval Rate (Base) | Mean PD (LGB) | Approval Rate (LGB)")
    print("  " + "-" * 75)
    
    for group_val in sorted(results_base.keys()):
        base_pd = results_base[group_val]['mean_pd']
        base_ar = results_base[group_val]['approval_rate']
        lgb_pd = results_lgb[group_val]['mean_pd']
        lgb_ar = results_lgb[group_val]['approval_rate']
        
        print(f"  {str(group_val):15s} | {base_pd:13.4f} | {base_ar:19.1%} | {lgb_pd:13.4f} | {lgb_ar:19.1%}")
    
    print("\n  Interpretation:")
    print("  - If higher PD → lower approval rate: Policy-driven (expected)")
    print("  - If similar PD but different approval rates: Model-driven (potential issue)")
    print("  - If different PD but similar approval rates: Threshold may be too permissive/strict")

In [ ]:
# Stability across temporal splits
print("=== STABILITY ACROSS TEMPORAL SPLITS ===\n")

# Get predictions for all splits
X_train_stable = train_df[approved_features].copy()
X_val_stable = val_df[approved_features].copy()

# Baseline model
X_train_base_scaled = scaler_base.transform(X_train_stable)
X_val_base_scaled = scaler_base.transform(X_val_stable)
X_train_base_scaled_df = pd.DataFrame(X_train_base_scaled, columns=approved_features, index=X_train_stable.index)
X_val_base_scaled_df = pd.DataFrame(X_val_base_scaled, columns=approved_features, index=X_val_stable.index)

y_train_base_proba = model_base.predict_proba(X_train_base_scaled_df)[:, 1]
y_val_base_proba = model_base.predict_proba(X_val_base_scaled_df)[:, 1]

# LightGBM
y_train_lgb_proba = model_lgb.predict(X_train_stable, num_iteration=model_lgb.best_iteration)
y_val_lgb_proba = model_lgb.predict(X_val_stable, num_iteration=model_lgb.best_iteration)

# Compute approval rates across splits
def compute_approval_rate_by_split(pd_values, split_name):
    approval_rate = (pd_values < pd_threshold).mean()
    return approval_rate

stability_data = {
    'Split': ['Train', 'Validation', 'Test'],
    'Baseline Approval Rate': [
        compute_approval_rate_by_split(y_train_base_proba, 'Train'),
        compute_approval_rate_by_split(y_val_base_proba, 'Val'),
        compute_approval_rate_by_split(y_test_base_proba, 'Test')
    ],
    'LightGBM Approval Rate': [
        compute_approval_rate_by_split(y_train_lgb_proba, 'Train'),
        compute_approval_rate_by_split(y_val_lgb_proba, 'Val'),
        compute_approval_rate_by_split(y_test_lgb_proba, 'Test')
    ],
    'Baseline Mean PD': [
        y_train_base_proba.mean(),
        y_val_base_proba.mean(),
        y_test_base_proba.mean()
    ],
    'LightGBM Mean PD': [
        y_train_lgb_proba.mean(),
        y_val_lgb_proba.mean(),
        y_test_lgb_proba.mean()
    ]
}

stability_df = pd.DataFrame(stability_data)
print(stability_df.to_string(index=False))

# Compute volatility (coefficient of variation)
base_ar_volatility = np.std(stability_df['Baseline Approval Rate']) / np.mean(stability_df['Baseline Approval Rate'])
lgb_ar_volatility = np.std(stability_df['LightGBM Approval Rate']) / np.mean(stability_df['LightGBM Approval Rate'])

print(f"\nApproval Rate Volatility (CV):")
print(f"  Baseline: {base_ar_volatility:.4f}")
print(f"  LightGBM: {lgb_ar_volatility:.4f}")
print(f"  {'Baseline more stable' if base_ar_volatility < lgb_ar_volatility else 'LightGBM more stable'}")

In [ ]:
# Distributional stress testing: higher-risk tail
print("\n=== DISTRIBUTIONAL STRESS TESTING ===\n")

# Identify high-risk tail (top 10% by PD)
high_risk_threshold_base = np.percentile(y_test_base_proba, 90)
high_risk_threshold_lgb = np.percentile(y_test_lgb_proba, 90)

high_risk_base = test_df[y_test_base_proba >= high_risk_threshold_base].copy()
high_risk_lgb = test_df[y_test_lgb_proba >= high_risk_threshold_lgb].copy()

print(f"High-risk tail (top 10% by PD):")
print(f"  Baseline: {len(high_risk_base):,} samples (PD >= {high_risk_threshold_base:.4f})")
print(f"  LightGBM: {len(high_risk_lgb):,} samples (PD >= {high_risk_threshold_lgb:.4f})")

# Compute metrics under stress
def compute_stress_metrics(df_stress, pd_values_stress, model_name):
    df_stress = df_stress.copy()
    df_stress['PD'] = pd_values_stress
    df_stress['EAD'] = df_stress['LIMIT_BAL']
    df_stress['approved'] = df_stress['PD'] < pd_threshold
    df_stress['expected_loss'] = df_stress['PD'] * df_stress['EAD'] * LGD_BASELINE
    
    approved = df_stress[df_stress['approved']]
    
    return {
        'approval_rate': df_stress['approved'].mean(),
        'mean_pd': df_stress['PD'].mean(),
        'mean_pd_approved': approved['PD'].mean() if len(approved) > 0 else 0,
        'total_expected_loss': approved['expected_loss'].sum() if len(approved) > 0 else 0,
        'default_rate_approved': approved[target_col].mean() if len(approved) > 0 else 0
    }

# Get PD values for high-risk groups
high_risk_base_indices = high_risk_base.index
high_risk_base_pd = y_test_base_proba[test_df.index.get_indexer(high_risk_base_indices)]

high_risk_lgb_indices = high_risk_lgb.index
high_risk_lgb_pd = y_test_lgb_proba[test_df.index.get_indexer(high_risk_lgb_indices)]

stress_base = compute_stress_metrics(high_risk_base, high_risk_base_pd, 'Baseline')
stress_lgb = compute_stress_metrics(high_risk_lgb, high_risk_lgb_pd, 'LightGBM')

# Compare with full test set
full_base = compute_stress_metrics(test_df, y_test_base_proba, 'Baseline')
full_lgb = compute_stress_metrics(test_df, y_test_lgb_proba, 'LightGBM')

print("\nStress Test Results (High-Risk Tail vs Full Test Set):")
print("\nBaseline Model:")
print(f"  Full Test Set:")
print(f"    Approval Rate: {full_base['approval_rate']:.1%}")
print(f"    Mean PD: {full_base['mean_pd']:.4f}")
print(f"    Total EL: ${full_base['total_expected_loss']:,.0f}")
print(f"  High-Risk Tail:")
print(f"    Approval Rate: {stress_base['approval_rate']:.1%}")
print(f"    Mean PD: {stress_base['mean_pd']:.4f}")
print(f"    Total EL: ${stress_base['total_expected_loss']:,.0f}")
print(f"    Change: {((stress_base['approval_rate'] - full_base['approval_rate']) / full_base['approval_rate']) * 100:+.1f}% approval rate")

print("\nLightGBM Model:")
print(f"  Full Test Set:")
print(f"    Approval Rate: {full_lgb['approval_rate']:.1%}")
print(f"    Mean PD: {full_lgb['mean_pd']:.4f}")
print(f"    Total EL: ${full_lgb['total_expected_loss']:,.0f}")
print(f"  High-Risk Tail:")
print(f"    Approval Rate: {stress_lgb['approval_rate']:.1%}")
print(f"    Mean PD: {stress_lgb['mean_pd']:.4f}")
print(f"    Total EL: ${stress_lgb['total_expected_loss']:,.0f}")
print(f"    Change: {((stress_lgb['approval_rate'] - full_lgb['approval_rate']) / full_lgb['approval_rate']) * 100:+.1f}% approval rate")

In [ ]:
# Threshold perturbation analysis
print("\n=== THRESHOLD PERTURBATION ANALYSIS ===\n")

# Test different threshold values
thresholds = np.arange(0.02, 0.08, 0.01)

perturbation_results = []

for threshold in thresholds:
    # Baseline
    approved_base = (y_test_base_proba < threshold).sum()
    approval_rate_base = (y_test_base_proba < threshold).mean()
    
    approved_base_df = test_df[y_test_base_proba < threshold].copy()
    approved_base_df['PD'] = y_test_base_proba[y_test_base_proba < threshold]
    approved_base_df['EAD'] = approved_base_df['LIMIT_BAL']
    approved_base_df['expected_loss'] = approved_base_df['PD'] * approved_base_df['EAD'] * LGD_BASELINE
    total_el_base = approved_base_df['expected_loss'].sum()
    default_rate_base = approved_base_df[target_col].mean() if len(approved_base_df) > 0 else 0
    
    # LightGBM
    approved_lgb = (y_test_lgb_proba < threshold).sum()
    approval_rate_lgb = (y_test_lgb_proba < threshold).mean()
    
    approved_lgb_df = test_df[y_test_lgb_proba < threshold].copy()
    approved_lgb_df['PD'] = y_test_lgb_proba[y_test_lgb_proba < threshold]
    approved_lgb_df['EAD'] = approved_lgb_df['LIMIT_BAL']
    approved_lgb_df['expected_loss'] = approved_lgb_df['PD'] * approved_lgb_df['EAD'] * LGD_BASELINE
    total_el_lgb = approved_lgb_df['expected_loss'].sum()
    default_rate_lgb = approved_lgb_df[target_col].mean() if len(approved_lgb_df) > 0 else 0
    
    perturbation_results.append({
        'Threshold': threshold,
        'Approval Rate (Base)': approval_rate_base,
        'Approval Rate (LGB)': approval_rate_lgb,
        'Total EL (Base)': total_el_base,
        'Total EL (LGB)': total_el_lgb,
        'Default Rate (Base)': default_rate_base,
        'Default Rate (LGB)': default_rate_lgb
    })

perturbation_df = pd.DataFrame(perturbation_results)

print("Decision Volatility Under Threshold Perturbations:")
print(perturbation_df.to_string(index=False))

# Compute sensitivity (change in approval rate per 0.01 change in threshold)
baseline_sensitivity = np.abs(np.diff(perturbation_df['Approval Rate (Base)'])).mean() / 0.01
lgb_sensitivity = np.abs(np.diff(perturbation_df['Approval Rate (LGB)'])).mean() / 0.01

print(f"\nSensitivity (approval rate change per 0.01 threshold change):")
print(f"  Baseline: {baseline_sensitivity:.2%}")
print(f"  LightGBM: {lgb_sensitivity:.2%}")
print(f"  {'Baseline more sensitive' if baseline_sensitivity > lgb_sensitivity else 'LightGBM more sensitive'}")

# Visualize threshold sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(perturbation_df['Threshold'], perturbation_df['Approval Rate (Base)'], 'o-', label='Baseline', alpha=0.7)
axes[0].plot(perturbation_df['Threshold'], perturbation_df['Approval Rate (LGB)'], 's-', label='LightGBM', alpha=0.7)
axes[0].axvline(pd_threshold, color='r', linestyle='--', label=f'Policy Threshold ({pd_threshold:.3f})')
axes[0].set_xlabel('PD Threshold')
axes[0].set_ylabel('Approval Rate')
axes[0].set_title('Approval Rate vs Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(perturbation_df['Threshold'], perturbation_df['Total EL (Base)'], 'o-', label='Baseline', alpha=0.7)
axes[1].plot(perturbation_df['Threshold'], perturbation_df['Total EL (LGB)'], 's-', label='LightGBM', alpha=0.7)
axes[1].axvline(pd_threshold, color='r', linestyle='--', label=f'Policy Threshold ({pd_threshold:.3f})')
axes[1].set_xlabel('PD Threshold')
axes[1].set_ylabel('Total Expected Loss ($)')
axes[1].set_title('Expected Loss vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Risk Interpretation

Explicitly discuss:
- Where the policy is brittle
- Where fairness risks emerge
- What constraints would be required in production

In [ ]:
print("=== RISK INTERPRETATION ===\n")

print("1. POLICY BRITTLENESS\n")
print("-" * 60)
print("""
**Where the policy is brittle**:

a) **Threshold sensitivity**: 
   - Small changes in PD threshold cause significant changes in approval rates
   - This makes the policy vulnerable to calibration drift
   - Requires continuous monitoring of PD distributions

b) **Temporal instability**:
   - Approval rates vary across train/val/test splits
   - Indicates potential distribution shift
   - Policy may not generalize to future data

c) **High-risk tail vulnerability**:
   - Small number of high-risk loans can drive total expected loss
   - Policy may be too permissive for edge cases
   - Requires additional safeguards for extreme PD values

d) **Model disagreement**:
   - Baseline and LightGBM may disagree on approval decisions
   - Creates uncertainty in production
   - Requires consensus mechanism or model selection criteria
""")

print("\n2. FAIRNESS RISKS\n")
print("-" * 60)
print("""
**Where fairness risks emerge**:

a) **Approval rate disparities**:
   - Different groups may have systematically different approval rates
   - If not justified by risk differences, this is discriminatory
   - Requires group-level monitoring and constraints

b) **Proxy-based groups**:
   - Sex, age, education are imperfect proxies for protected attributes
   - May miss intersectional effects
   - Requires careful interpretation and legal review

c) **Model-driven disparities**:
   - If models estimate PD differently across groups for non-risk reasons
   - This creates unfairness even if policy is risk-neutral
   - Requires group-level calibration checks

d) **Access vs Risk tradeoff**:
   - Stricter thresholds reduce risk but limit access
   - May disproportionately affect certain groups
   - Requires explicit business decision on acceptable tradeoffs
""")

print("\n3. PRODUCTION CONSTRAINTS REQUIRED\n")
print("-" * 60)
print("""
**Constraints that would be required in production**:

a) **Fairness constraints**:
   - Maximum approval rate disparity between groups (e.g., 80% rule)
   - Minimum approval rate for any group
   - Regular audits of group-level outcomes

b) **Stability constraints**:
   - Maximum change in approval rate between monitoring periods
   - Alert if PD distribution shifts significantly
   - Require recalibration if drift detected

c) **Risk constraints**:
   - Maximum expected loss per approved loan
   - Maximum default rate among approved loans
   - Portfolio-level risk limits

d) **Model governance**:
   - Document all model assumptions and limitations
   - Regular retraining schedule
   - A/B testing for model updates
   - Explainability requirements for rejected applications

e) **Monitoring requirements**:
   - Real-time tracking of approval rates by group
   - Weekly reports on expected loss and default rates
   - Quarterly fairness audits
   - Annual model validation
""")

print("\n4. KEY TRADEOFFS\n")
print("-" * 60)
print("""
**Risk Reduction vs Access**:
- Stricter thresholds reduce expected loss but limit credit access
- Must balance financial risk with business objectives
- May require different thresholds for different segments

**Model Complexity vs Explainability**:
- LightGBM may capture non-linearities but is harder to explain
- Logistic regression is more interpretable but may miss patterns
- Regulatory requirements may favor simpler models

**Fairness vs Performance**:
- Enforcing fairness constraints may reduce model discrimination
- May increase expected loss to ensure equitable access
- Requires explicit business decision on acceptable tradeoffs

**Stability vs Adaptability**:
- Stable models are less sensitive to distribution shift
- But may fail to adapt to changing risk patterns
- Requires careful balance and monitoring
""")

print("\n5. RECOMMENDATIONS\n")
print("-" * 60)
print("""
**For production deployment**:

1. **Implement group-level monitoring**: Track approval rates, expected loss, and default rates by demographic groups

2. **Set fairness constraints**: Define maximum acceptable disparities (e.g., 80% rule for approval rates)

3. **Establish stability thresholds**: Alert if approval rate changes by more than X% between periods

4. **Create risk limits**: Set maximum expected loss per loan and portfolio-level limits

5. **Document model limitations**: Clearly state proxy-based group analysis limitations and assumptions

6. **Regular audits**: Conduct quarterly fairness audits and annual model validation

7. **Explainability requirements**: Provide SHAP values or feature contributions for rejected applications

8. **Model selection criteria**: Define when to use baseline vs LightGBM based on stability, fairness, and performance

9. **Threshold review process**: Establish regular review of PD thresholds based on business conditions

10. **Integration with drift detection**: Connect to existing drift detection system for continuous monitoring
""")

## Summary

**Fairness, Stability & Stress Testing completed**:

1. **Group definitions**: Identified subgroups using sex, age, education, and credit limit (with acknowledged limitations)

2. **Fairness diagnostics**: Computed approval rates, expected loss, and default rates by group for both models

3. **Disparity assessment**: Quantified absolute and relative disparities, distinguished policy-driven vs model-driven differences

4. **Stability & stress testing**: Evaluated robustness under temporal splits, distributional stress, and threshold perturbations

5. **Risk interpretation**: Identified policy brittleness, fairness risks, and production constraints required

**Key findings**:
- Approval rate disparities exist across groups and may be policy-driven or model-driven
- Policy is sensitive to threshold changes and temporal shifts
- High-risk tail requires additional safeguards
- Production deployment requires fairness constraints, stability monitoring, and risk limits

**Next steps**:
- This analysis provides evidence for production constraints and monitoring requirements
- Business stakeholders must decide on acceptable tradeoffs (risk vs access, fairness vs performance)
- Integration with drift detection system for continuous monitoring